# TuringSQL 1/2: prepare BIRD train (CPU session, no GPU quota)

BIRD's training databases are 32 GB unzipped, too big for the laptop. This notebook downloads the official
`train.zip`, processes the databases **one at a time** (extract, check every gold query, write the schema
text, delete), then builds the fine-tuning files from Spider + BIRD.

Output: `/kaggle/working/TuringSQL/data/processed/` (sft_train.jsonl, sft_val.jsonl and reports). Save the
notebook version, then attach its output to notebook 2.

Settings: Accelerator **None**, Internet **On**.

In [ ]:
# Code + data are uploaded as a private Kaggle dataset built by scripts/package_for_kaggle.py
import glob
import json
import os
import subprocess
import sys
import zipfile
import shutil

WORK = "/kaggle/working/TuringSQL"
if not os.path.isdir(WORK):
    # Kaggle usually unzips uploaded archives; handle both the zip and the unpacked folder.
    zips = glob.glob("/kaggle/input/**/turingsql_kaggle.zip", recursive=True)
    trees = [os.path.dirname(os.path.dirname(p))
             for p in glob.glob("/kaggle/input/**/turingsql/__init__.py", recursive=True)]

    def built(tree):  # newest package wins; copies without a stamp (older code) lose
        f = os.path.join(tree, "PACKAGE_INFO.json")
        return json.load(open(f))["built_utc"] if os.path.isfile(f) else ""

    trees.sort(key=built, reverse=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall(WORK)
    elif trees:
        print("code from", trees[0], built(trees[0]) or "(no build stamp)")
        shutil.copytree(trees[0], WORK)
    else:
        raise FileNotFoundError("attach the turingsql-kaggle dataset (dist/turingsql_kaggle.zip)")
os.chdir(WORK)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

In [ ]:
!pip -q install "sqlglot>=25" "transformers>=4.45,<5"
!df -h /kaggle/working /tmp | cat

In [ ]:
# Official BIRD train set (CC BY-SA 4.0), about 8.9 GB
!mkdir -p /tmp/bird
if not os.path.isfile("/tmp/bird/train.zip"):
    !wget -q --show-progress -O /tmp/bird/train.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip
!ls -la /tmp/bird

In [ ]:
!python data_prep/prepare_public.py bird-train --zip /tmp/bird/train.zip --workdir /tmp/bird/work

In [ ]:
# Tokenizer only (small download), used to drop examples longer than 2,048 tokens
!python data_prep/build_sft.py --sources spider-train bird-train --tokenizer Qwen/Qwen2.5-Coder-1.5B-Instruct

In [ ]:
# Keep only the small processed files as notebook output; free the scratch space
!rm -rf /tmp/bird/work
!ls -la data/processed && du -sh data/processed